In [1]:
import pandas as pd

# List of CO2 files to include (excluding co2_2024)
co2_files = [
    'co2_2018.csv',
    'co2_2019.csv',
    'co2_2020.csv',
    'co2_2021.csv',
    'co2_2022.csv',
    'co2_2023.csv'
]

# Load the training dataset
train_df = pd.read_csv('train_with_gas.csv')

# Ensure the 'ds' column is in datetime format
train_df['ds'] = pd.to_datetime(train_df['ds'])

# Combine the CO2 data from the specified files
co2_dataframes = []

for co2_file in co2_files:
    co2_df = pd.read_csv(co2_file)
    co2_df = co2_df.rename(columns={"Day": "date", "CO2 Emission Allowances, Auction DE": "co2_emissions"})  # Adjust column names if needed
    co2_df['date'] = pd.to_datetime(co2_df['date'], format='%d.%m.%Y')  # Convert 'date' column to datetime
    co2_dataframes.append(co2_df)

# Combine all CO2 datasets into a single DataFrame
all_co2_df = pd.concat(co2_dataframes, ignore_index=True)

# Handle missing values with interpolation and forward/backward filling
all_co2_df['co2_emissions'] = all_co2_df['co2_emissions'].interpolate(method='linear')  # Interpolate missing values
all_co2_df['co2_emissions'] = all_co2_df['co2_emissions'].ffill().bfill()  # Forward and backward fill



In [2]:
# Merge the CO2 emissions data into the training dataset
train_df['date'] = train_df['ds'].dt.date  # Extract the date from 'ds'
train_df['date'] = pd.to_datetime(train_df['date'])  # Ensure the date is in datetime64[ns] type
train_df = pd.merge(train_df, all_co2_df, how='left', left_on='date', right_on='date')

# Drop the temporary 'date' column
train_df = train_df.drop(columns=['date'])

# Save the updated training dataset to a new CSV file
train_df.to_csv('train_with_co2.csv', index=False)

print("All CO2 data successfully added to the training dataset, with missing values handled.")

All CO2 data successfully added to the training dataset, with missing values handled.
